<a href="https://colab.research.google.com/github/flahbocchino/cardioia-fase5-assistente-paciente/blob/main/ir_alem_2_rpa_ia_dados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CardioIA — Fase 5 — Ir Além 2
## Automação Inteligente com RPA, IA e Dados Híbridos

**Objetivo:** simular um robô (RPA) que lê periodicamente dados clínicos simulados
(pressão arterial, frequência cardíaca, adesão ao tratamento) armazenados em um
**banco relacional (SQLite)**, aplica uma checagem por **regras clínicas** e um
**modelo de IA (`IsolationForest`)** para detectar padrões anômalos, e registra
alertas e logs de execução de forma rastreável em um **banco não relacional
(MongoDB Atlas)**.

**Fluxo:** `SQLite (dados dos pacientes) → Robô (regras + IA) → MongoDB (logs + alertas)`

**Notebook único e autocontido** — não precisa editar nenhuma célula manualmente
além de configurar o secret `MONGO_URI` (passo 2).

## 1. Instalação das dependências

In [13]:
!pip install pymongo scikit-learn pandas --quiet
print("Dependências instaladas.")


Dependências instaladas.


## 2. Configuração

**IMPORTANTE:** você precisa da sua *connection string* do MongoDB Atlas
(Connect → Drivers → Python). Troque `<senha>` pela senha real do usuário
que você criou em Database Access.

Nunca cole essa string com a senha real num arquivo que vai pro GitHub —
use a aba de **Secrets** do Colab (ícone de chave 🔑 na barra lateral) para
guardar `MONGO_URI` com segurança, com o toggle "Notebook access" ligado.

In [14]:
from google.colab import userdata

try:
    MONGO_URI = userdata.get("MONGO_URI")
except Exception:
    # Alternativa apenas para teste rápido — não deixe a senha real aqui no arquivo final:
    MONGO_URI = "mongodb+srv://usuario:<senha>@cluster0.xxxxx.mongodb.net/"

DB_SQLITE = "cardioia_pacientes.db"
NOME_DB_MONGO = "cardioia_fase5"
print("Configuração carregada.")


Configuração carregada.


## 3. Banco relacional (SQLite) — dados clínicos simulados

Cria a tabela `sinais_vitais`, com leituras periódicas simuladas de 10 pacientes:
pressão arterial (sistólica/diastólica), frequência cardíaca e adesão ao
tratamento (%). Cerca de 8% das leituras são geradas propositalmente fora da
faixa normal, para que o robô tenha o que detectar.

In [15]:
import sqlite3
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

np.random.seed(42)

conn = sqlite3.connect(DB_SQLITE)
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS sinais_vitais")
cursor.execute("""
CREATE TABLE sinais_vitais (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    paciente_id INTEGER NOT NULL,
    timestamp TEXT NOT NULL,
    pressao_sistolica INTEGER NOT NULL,
    pressao_diastolica INTEGER NOT NULL,
    frequencia_cardiaca INTEGER NOT NULL,
    adesao_tratamento REAL NOT NULL,
    processado INTEGER DEFAULT 0
)
""")

n_pacientes = 10
leituras_por_paciente = 20
base_time = datetime.now() - timedelta(days=5)

linhas = []
for paciente_id in range(1, n_pacientes + 1):
    for i in range(leituras_por_paciente):
        ts = base_time + timedelta(hours=6 * i, minutes=paciente_id)

        # valores normais como padrão
        sistolica = int(np.random.normal(120, 8))
        diastolica = int(np.random.normal(80, 6))
        fc = int(np.random.normal(75, 8))
        adesao = round(np.random.uniform(70, 100), 1)

        # ~8% de chance de leitura anômala proposital (simula evento clínico real)
        if np.random.rand() < 0.08:
            tipo_anomalia = np.random.choice(["hipertensao", "taquicardia", "bradicardia", "baixa_adesao"])
            if tipo_anomalia == "hipertensao":
                sistolica = int(np.random.normal(165, 10))
                diastolica = int(np.random.normal(100, 8))
            elif tipo_anomalia == "taquicardia":
                fc = int(np.random.normal(135, 10))
            elif tipo_anomalia == "bradicardia":
                fc = int(np.random.normal(38, 5))
            elif tipo_anomalia == "baixa_adesao":
                adesao = round(np.random.uniform(10, 40), 1)

        linhas.append((paciente_id, ts.isoformat(), sistolica, diastolica, fc, adesao))

cursor.executemany("""
INSERT INTO sinais_vitais (paciente_id, timestamp, pressao_sistolica, pressao_diastolica, frequencia_cardiaca, adesao_tratamento)
VALUES (?, ?, ?, ?, ?, ?)
""", linhas)

conn.commit()
print(f"{len(linhas)} leituras simuladas inseridas no SQLite ({DB_SQLITE}).")
pd.read_sql("SELECT * FROM sinais_vitais LIMIT 5", conn)


200 leituras simuladas inseridas no SQLite (cardioia_pacientes.db).


,id,paciente_id,timestamp,pressao_sistolica,pressao_diastolica,frequencia_cardiaca,adesao_tratamento,processado
0,1,1,2026-09-05T14:44:30.780679,123,79,80,74.7,0
1,2,1,2026-09-05T20:44:30.780679,132,89,81,70.6,0
2,3,1,2026-09-06T02:44:30.780679,116,83,71,79.1,0
3,4,1,2026-09-06T08:44:30.780679,116,69,70,88.4,0
4,5,1,2026-09-06T14:44:30.780679,112,71,86,76.0,0


## 4. Conexão com o banco não relacional (MongoDB Atlas)

Duas coleções:
- `logs_execucao`: registra **cada rodada** do robô (rastreabilidade do processo em si).
- `alertas`: registra apenas os eventos que exigem atenção (rastreabilidade clínica).

In [16]:
import pymongo
from datetime import datetime

cliente_mongo = pymongo.MongoClient(MONGO_URI)
db_mongo = cliente_mongo[NOME_DB_MONGO]

colecao_logs = db_mongo["logs_execucao"]
colecao_alertas = db_mongo["alertas"]

# Teste de conexão
print("Conectado ao MongoDB Atlas. Bancos disponíveis:", cliente_mongo.list_database_names())


Conectado ao MongoDB Atlas. Bancos disponíveis: ['sample_mflix', 'admin', 'local']


## 5. O robô (RPA): leitura periódica + regras clínicas + IA

O robô simula execuções periódicas. A cada rodada:
1. Lê do SQLite apenas as leituras ainda **não processadas** (`processado = 0`).
2. Aplica **regras clínicas fixas** (limites conhecidos na literatura, ex.: FC > 120 ou < 40 bpm).
3. Aplica um modelo de IA (`IsolationForest`) sobre o lote, para capturar combinações
   fora do padrão que uma regra isolada não pegaria (ex.: pressão e FC levemente
   alteradas ao mesmo tempo).
4. Marca as leituras como processadas.
5. Registra o resultado da rodada em `logs_execucao` e, se houver risco, grava em `alertas`.

In [17]:
from sklearn.ensemble import IsolationForest

def aplicar_regras_clinicas(linha):
    """Regras fixas de triagem. Retorna lista de motivos disparados (vazia se normal)."""
    motivos = []
    if linha["frequencia_cardiaca"] > 120:
        motivos.append(f"Taquicardia (FC={linha['frequencia_cardiaca']} bpm)")
    if linha["frequencia_cardiaca"] < 40:
        motivos.append(f"Bradicardia (FC={linha['frequencia_cardiaca']} bpm)")
    if linha["pressao_sistolica"] > 140 or linha["pressao_diastolica"] > 90:
        motivos.append(f"Pressão elevada ({linha['pressao_sistolica']}/{linha['pressao_diastolica']} mmHg)")
    if linha["adesao_tratamento"] < 50:
        motivos.append(f"Baixa adesão ao tratamento ({linha['adesao_tratamento']}%)")
    return motivos


def executar_ciclo_robo():
    """Executa uma rodada completa do robô e retorna um resumo."""
    conn = sqlite3.connect(DB_SQLITE)
    df = pd.read_sql("SELECT * FROM sinais_vitais WHERE processado = 0", conn)

    inicio = datetime.now()

    if df.empty:
        log = {
            "timestamp": inicio.isoformat(),
            "status": "sem_dados_novos",
            "leituras_processadas": 0,
            "alertas_gerados": 0
        }
        colecao_logs.insert_one(log)
        conn.close()
        return log

    # --- Camada 1: regras clínicas ---
    df["motivos_regra"] = df.apply(aplicar_regras_clinicas, axis=1)
    df["risco_regra"] = df["motivos_regra"].apply(lambda m: len(m) > 0)

    # --- Camada 2: IA (detecção de anomalia multivariada) ---
    features = df[["pressao_sistolica", "pressao_diastolica", "frequencia_cardiaca", "adesao_tratamento"]]
    modelo = IsolationForest(contamination=0.1, random_state=42)
    df["anomalia_ia"] = modelo.fit_predict(features) == -1  # -1 = anomalia

    alertas_gerados = 0
    for _, linha in df.iterrows():
        if linha["risco_regra"] or linha["anomalia_ia"]:
            motivo_final = list(linha["motivos_regra"])
            if linha["anomalia_ia"] and not linha["risco_regra"]:
                motivo_final.append("Padrão multivariado fora do esperado (detectado por IA)")

            alerta = {
                "timestamp": datetime.now().isoformat(),
                "paciente_id": int(linha["paciente_id"]),
                "leitura_timestamp": linha["timestamp"],
                "pressao_sistolica": int(linha["pressao_sistolica"]),
                "pressao_diastolica": int(linha["pressao_diastolica"]),
                "frequencia_cardiaca": int(linha["frequencia_cardiaca"]),
                "adesao_tratamento": float(linha["adesao_tratamento"]),
                "motivos": motivo_final,
                "origem": "regra" if linha["risco_regra"] else "ia"
            }
            colecao_alertas.insert_one(alerta)
            alertas_gerados += 1

    # marca como processado no SQLite
    ids = df["id"].tolist()
    conn.executemany("UPDATE sinais_vitais SET processado = 1 WHERE id = ?", [(i,) for i in ids])
    conn.commit()
    conn.close()

    log = {
        "timestamp": inicio.isoformat(),
        "status": "concluido",
        "leituras_processadas": len(df),
        "alertas_gerados": alertas_gerados,
        "duracao_segundos": (datetime.now() - inicio).total_seconds()
    }
    colecao_logs.insert_one(log)
    return log


## 6. Executar o robô

Simula rodadas periódicas (na vida real seria um agendador/cron; aqui simulamos
chamando a função algumas vezes, em lotes, para mostrar o ciclo completo:
leitura → processamento → log → alerta).

In [18]:
for rodada in range(1, 4):
    resultado = executar_ciclo_robo()
    print(f"Rodada {rodada}: {resultado}")


Rodada 1: {'timestamp': '2026-09-10T14:43:34.105964', 'status': 'concluido', 'leituras_processadas': 200, 'alertas_gerados': 28, 'duracao_segundos': 6.620105, '_id': ObjectId('6aa2c21c8c66138bb95f7779')}
Rodada 2: {'timestamp': '2026-09-10T14:43:40.989226', 'status': 'sem_dados_novos', 'leituras_processadas': 0, 'alertas_gerados': 0, '_id': ObjectId('6aa2c21c8c66138bb95f777a')}
Rodada 3: {'timestamp': '2026-09-10T14:43:41.224403', 'status': 'sem_dados_novos', 'leituras_processadas': 0, 'alertas_gerados': 0, '_id': ObjectId('6aa2c21d8c66138bb95f777b')}


## 7. Verificação de rastreabilidade\n\nConsulta os alertas e logs gravados no MongoDB, comprovando que cada ação do robô fica registrada.

In [19]:
print("=== Logs de execução ===")
for log in colecao_logs.find().sort("timestamp", -1).limit(5):
    print(log)

print("\n=== Alertas gerados (mais recentes) ===")
for alerta in colecao_alertas.find().sort("timestamp", -1).limit(5):
    print(alerta)

print("\nTotal de alertas no banco:", colecao_alertas.count_documents({}))
print("Total de execuções registradas:", colecao_logs.count_documents({}))


=== Logs de execução ===
{'_id': ObjectId('6aa2c21d8c66138bb95f777b'), 'timestamp': '2026-09-10T14:43:41.224403', 'status': 'sem_dados_novos', 'leituras_processadas': 0, 'alertas_gerados': 0}
{'_id': ObjectId('6aa2c21c8c66138bb95f777a'), 'timestamp': '2026-09-10T14:43:40.989226', 'status': 'sem_dados_novos', 'leituras_processadas': 0, 'alertas_gerados': 0}
{'_id': ObjectId('6aa2c21c8c66138bb95f7779'), 'timestamp': '2026-09-10T14:43:34.105964', 'status': 'concluido', 'leituras_processadas': 200, 'alertas_gerados': 28, 'duracao_segundos': 6.620105}

=== Alertas gerados (mais recentes) ===
{'_id': ObjectId('6aa2c21c8c66138bb95f7778'), 'timestamp': '2026-09-10T14:43:40.483548', 'paciente_id': 10, 'leitura_timestamp': '2026-09-08T20:53:30.780679', 'pressao_sistolica': 125, 'pressao_diastolica': 92, 'frequencia_cardiaca': 72, 'adesao_tratamento': 92.8, 'motivos': ['Pressão elevada (125/92 mmHg)'], 'origem': 'regra'}
{'_id': ObjectId('6aa2c21c8c66138bb95f7777'), 'timestamp': '2026-09-10T14:43

## 8. Estrutura dos bancos utilizados (para o relatório técnico)

**Banco relacional — SQLite (`cardioia_pacientes.db`)**
Tabela `sinais_vitais`:

| Coluna | Tipo | Descrição |
|---|---|---|
| id | INTEGER (PK) | identificador da leitura |
| paciente_id | INTEGER | identifica o paciente simulado |
| timestamp | TEXT | data/hora da leitura |
| pressao_sistolica | INTEGER | mmHg |
| pressao_diastolica | INTEGER | mmHg |
| frequencia_cardiaca | INTEGER | bpm |
| adesao_tratamento | REAL | % de adesão simulada |
| processado | INTEGER | flag usada pelo robô (0/1) |

**Banco não relacional — MongoDB Atlas (`cardioia_fase5`)**

- `logs_execucao`: um documento por rodada do robô (timestamp, status, quantidade processada, quantidade de alertas, duração).
- `alertas`: um documento por evento de risco (paciente, valores, motivos disparados, se veio de regra ou de IA).

*(Este bloco pode ser copiado direto para o relatório técnico do Ir Além 2.)*